# Part 3 & 4 — Authenticity Index Analysis

Explore the Organizational Authenticity Index across companies, sectors, and time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

index_df = pd.read_csv('../data/processed/part3_authenticity_index.csv')
valid = index_df[index_df['has_both_sources'] & index_df['authenticity_score'].notna()].copy()
print(f'{len(valid)} valid company-year observations')
valid.describe()

In [ ]:
# Company-level mean scores
company_means = valid.groupby(['ticker', 'company_name', 'sector'])['authenticity_score'].mean().reset_index()
company_means = company_means.sort_values('authenticity_score', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
colors = {'Technology': '#4e79a7', 'Financials': '#f28e2b', 'Healthcare': '#59a14f',
          'Consumer Discretionary': '#e15759', 'Energy': '#76b7b2'}
bar_colors = [colors.get(s, 'gray') for s in company_means['sector']]
ax.barh(company_means['ticker'], company_means['authenticity_score'], color=bar_colors)
ax.axvline(x=valid['authenticity_score'].mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
ax.set_title('Mean Organizational Authenticity Score by Company (2016–2024)')
ax.set_xlabel('Authenticity Score (cosine similarity)')
plt.tight_layout()
plt.show()

In [ ]:
# Load and display summary statistics
summary = pd.read_csv('../data/outputs/part3_summary_statistics.csv')
summary

In [ ]:
# Sector trends over time
sector_year = valid.groupby(['sector', 'year'])['authenticity_score'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
for sector, grp in sector_year.groupby('sector'):
    ax.plot(grp['year'], grp['authenticity_score'], marker='o', label=sector, linewidth=2)
ax.set_title('Authenticity Score by Sector Over Time')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Authenticity Score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# For a selected company, show theme-level gap between stated and lived values
TICKER = 'JNJ'  # Change to any ticker

vectors = pd.read_csv('../data/interim/part3/theme_vectors.csv')
co = vectors[vectors['ticker'] == TICKER].sort_values('year')

themes = ['innovation', 'customers_or_patients', 'employees', 'diversity_equity_inclusion',
          'sustainability_environment', 'community_social_impact', 'ethics_integrity',
          'governance_accountability', 'financial_performance', 'safety_quality']

year_plot = co[co['year'] == 2023].iloc[0] if 2023 in co['year'].values else co.iloc[-1]
about_scores = [year_plot.get(f'about_{t}', 0) or 0 for t in themes]
proxy_scores = [year_plot.get(f'proxy_{t}', 0) or 0 for t in themes]

x = range(len(themes))
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - 0.2 for i in x], about_scores, 0.4, label='About Us (Stated)', color='steelblue', alpha=0.8)
ax.bar([i + 0.2 for i in x], proxy_scores, 0.4, label='Proxy (Lived)', color='coral', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels([t.replace('_', ' ') for t in themes], rotation=30, ha='right')
ax.set_title(f'{TICKER}: Stated vs. Lived Theme Emphasis ({int(year_plot["year"])})')
ax.set_ylabel('Theme Score (0–3)')
ax.legend()
plt.tight_layout()
plt.show()